# Fixed versus semantic transcript segmentation

This notebook compares the existing fixed 2–4 minute passages with local semantic boundaries. BGE-M3 scores topic changes from cached cue-aligned atomic blocks; Qwen and BGE then receive exactly the same passages within each segmentation mode.

All segmentation runs locally. No transcript text is sent to Gemini or another cloud API. Start with a few episodes and inspect the boundary plot before scaling to the full archive.

In [ ]:
import os
from dataclasses import replace
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from arvamusfestivali_transcripts.topic_analysis import (
    BgeM3Embedder,
    ChunkingConfig,
    DeterministicHashEmbedder,
    QwenEmbedder,
    SemanticSegmentationConfig,
    TopicModelConfig,
    build_atomic_blocks_many,
    chunk_episodes,
    embed_passages,
    evaluate_run,
    fit_topic_model,
    load_corpus,
    plot_semantic_boundaries,
    plot_topic_correspondence,
    plot_topic_sizes,
    segment_episodes_semantically,
    select_diverse_episodes,
)
from arvamusfestivali_transcripts.topic_analysis.cache import EmbeddingCache

pd.set_option("display.max_colwidth", 160)

## Configuration

In [ ]:
PROJECT_ROOT = Path(os.environ.get("TOPIC_ANALYSIS_PROJECT_ROOT", Path.cwd())).resolve()
if (
    not (PROJECT_ROOT / "data" / "transcripts").exists()
    and (PROJECT_ROOT.parent / "data" / "transcripts").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

YEAR = 2026
EPISODE_COUNT = 219
EXPLICIT_EPISODE_IDS: tuple[str, ...] = ()
SEGMENTATION_MODES = ("fixed", "semantic")
MODEL_KEYS = ("qwen", "bge")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZES = {"qwen": 2, "bge": 2}
CACHE_ROOT = PROJECT_ROOT / "data" / "topic-analysis" / "cache"
FIXED_CHUNKING = ChunkingConfig()
SEMANTIC_CONFIG = SemanticSegmentationConfig()
SEMANTIC_CACHE_CHUNKING = ChunkingConfig(
    min_seconds=90, target_seconds=300, max_seconds=600,
    min_words=1, max_words=1_000_000, overlap_seconds=0,
    merge_tail_seconds=0, max_merged_seconds=600,
)
TOPIC_MODEL = TopicModelConfig(random_state=42)
DRY_RUN_WITH_FAKE_EMBEDDINGS = False
RESULT_ROOT = Path(os.environ.get("TOPIC_ANALYSIS_RESULT_ROOT", PROJECT_ROOT / "results" / "topic-detail"))
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

## Load canonical recordings

In [ ]:
episodes = load_corpus(PROJECT_ROOT / "data" / "transcripts", YEAR)
if not episodes:
    raise ValueError(f"No validated transcripts found for {YEAR}")
selected = select_diverse_episodes(
    episodes, count=min(EPISODE_COUNT, len(episodes)), explicit_ids=EXPLICIT_EPISODE_IDS or None
)
display(pd.DataFrame([
    {
        "episode_id": episode.episode_id,
        "title": episode.title,
        "duration_minutes": episode.duration_seconds / 60,
        "duplicate_episode_ids": ", ".join(episode.duplicate_episode_ids),
    }
    for episode in selected
]))

## Configure local embedding adapters

In [ ]:
if DRY_RUN_WITH_FAKE_EMBEDDINGS:
    boundary_adapter = DeterministicHashEmbedder(dimension=32)
    boundary_adapter.key = "bge-boundary"
    boundary_adapter.model_id = "deterministic-hash-bge-boundary"
    adapters = {}
    for model_key in MODEL_KEYS:
        adapter = DeterministicHashEmbedder(dimension=32)
        adapter.key = model_key
        adapter.model_id = f"deterministic-hash-{model_key}"
        adapters[model_key] = adapter
    active_topic_model = TopicModelConfig(
        n_neighbors=3, n_components=2, min_cluster_size=2,
        min_samples=1, random_state=42,
    )
else:
    adapters = {
        "qwen": QwenEmbedder(device=DEVICE),
        "bge": BgeM3Embedder(device=DEVICE),
    }
    boundary_adapter = adapters["bge"]
    active_topic_model = TOPIC_MODEL

cache = EmbeddingCache(CACHE_ROOT)
print(f"Device: {DEVICE}; cache: {CACHE_ROOT}")

## Build fixed passages and semantic boundaries

In [ ]:
fixed_passages = chunk_episodes(selected, FIXED_CHUNKING)
atomic_blocks = build_atomic_blocks_many(selected, SEMANTIC_CONFIG)
boundary_embeddings = embed_passages(
    boundary_adapter,
    atomic_blocks,
    cache,
    BATCH_SIZES["bge"],
    chunking=SEMANTIC_CONFIG.atomic_chunking,
)
semantic_result = segment_episodes_semantically(
    selected,
    atomic_blocks,
    boundary_embeddings,
    SEMANTIC_CONFIG,
)
passage_sets = {
    "fixed": fixed_passages,
    "semantic": semantic_result.passages,
}

summary_rows = []
for mode, passages in passage_sets.items():
    durations = [(item.end_seconds - item.start_seconds) / 60 for item in passages]
    summary_rows.append({
        "mode": mode,
        "passages": len(passages),
        "median_minutes": pd.Series(durations).median(),
        "min_minutes": min(durations),
        "max_minutes": max(durations),
    })
display(pd.DataFrame(summary_rows))
print(
    f"Atomic BGE embeddings: {boundary_embeddings.embeddings.shape}; "
    f"{boundary_embeddings.cache_hits} cache hits"
)

## Inspect boundary scores before topic modelling

In [ ]:
first_episode = selected[0]
first_blocks = tuple(
    block for block in atomic_blocks if block.episode_id == first_episode.episode_id
)
first_boundaries = tuple(
    boundary
    for boundary in semantic_result.boundaries
    if boundary.episode_id == first_episode.episode_id
)
boundary_figures = [
    plot_semantic_boundaries(first_episode, first_blocks, first_boundaries)
]
display(boundary_figures[0])

## Embed identical passage sets with Qwen and BGE

In [ ]:
embedding_runs = {}
for mode in SEGMENTATION_MODES:
    passages = passage_sets[mode]
    cache_chunking = FIXED_CHUNKING if mode == "fixed" else SEMANTIC_CACHE_CHUNKING
    for model_key in MODEL_KEYS:
        result = embed_passages(
            adapters[model_key],
            passages,
            cache,
            BATCH_SIZES[model_key],
            chunking=cache_chunking,
        )
        embedding_runs[(mode, model_key)] = replace(
            result, model_key=f"{mode}:{model_key}"
        )
        print(
            f"{mode}:{model_key}: {result.embeddings.shape}, "
            f"{result.cache_hits} cache hits"
        )

## Fit BERTopic for every segmentation/model pair

In [ ]:
runs = {}
for mode in SEGMENTATION_MODES:
    for model_key in MODEL_KEYS:
        runs[(mode, model_key)] = fit_topic_model(
            passage_sets[mode],
            embedding_runs[(mode, model_key)],
            active_topic_model,
        )

metrics = []
for (mode, model_key), run in runs.items():
    metrics.append({
        "segmentation": mode,
        "embedding_model": model_key,
        **evaluate_run(run, embedding_runs[(mode, model_key)]),
    })
display(pd.DataFrame(metrics))

## Compare topic structure

In [ ]:
topic_size_figures = {
    key: plot_topic_sizes(run, passage_sets[key[0]])
    for key, run in runs.items()
}
correspondence_figures = {
    mode: plot_topic_correspondence(
        {
            f"{mode}:qwen": runs[(mode, "qwen")],
            f"{mode}:bge": runs[(mode, "bge")],
        },
        passage_sets[mode],
    )
    for mode in SEGMENTATION_MODES
}
for mode in SEGMENTATION_MODES:
    display(topic_size_figures[(mode, "qwen")])
    display(topic_size_figures[(mode, "bge")])
    display(correspondence_figures[mode])

In [ ]:
OUTPUT_DIR = RESULT_ROOT / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for (mode, model_key), run in runs.items():
    figure = plot_topic_sizes(run, passage_sets[mode])
    figure.update_traces(
        customdata=None,
        hovertemplate="%{x}: %{y} passages<extra></extra>",
    )

    path = OUTPUT_DIR / f"{mode}-{model_key}-topic-sizes.html"
    figure.write_html(path, include_plotlyjs=True)
    run.topic_info.to_csv(
        OUTPUT_DIR / f"{mode}-{model_key}-topics.csv",
        index=False,
    )

    print(f"Saved: {path}")

In [ ]:
for mode in SEGMENTATION_MODES:
    figure = plot_topic_correspondence(
        {
            f"{mode}:qwen": runs[(mode, "qwen")],
            f"{mode}:bge": runs[(mode, "bge")],
        },
        passage_sets[mode],
    )

    for trace in figure.data:
        if trace.type == "sankey":
            trace.link.customdata = None
            trace.link.hovertemplate = (
                "%{value} shared passages<extra></extra>"
            )

    path = OUTPUT_DIR / f"{mode}-qwen-bge-correspondence.html"
    figure.write_html(path, include_plotlyjs=True)
    figure.show(renderer="notebook_connected")

    print(f"Saved: {path}")

## Fine topics: preserve detail before merging

The recorded baseline had 137/178 fixed and 105/139 semantic topics (Qwen/BGE).
The previous exploratory configuration increased neighborhood and cluster size and found only
97 topics. Reducing to 40 then put 2,235 passages in one topic. That reduction is removed.

Compare EOM with **leaf** cluster selection, smaller minimum clusters and lower `min_samples`.
Leaf selection favors finer clusters ([HDBSCAN documentation](https://hdbscan.readthedocs.io/en/latest/parameter_selection.html#leaf-clustering)).
No topic count is imposed and no outliers are forcibly assigned. A small cluster can represent
an important part of just one talk; it does not need support from multiple talks.

Keep the cached segmentation unchanged first, so clustering improvements can be isolated.
Semantic passages can be ten minutes long and contain multiple subjects. For a separate
segmentation experiment try `min_segment_seconds=60`, `max_segment_seconds=240`,
`boundary_quantile=0.70` above, then rerun; changed passages need new embeddings.
Stopwords affect topic descriptions, not semantic embeddings. Preserve original spelling and
accents; inspect ASR errors rather than automatically stripping characters.

BERTopic must use `language="multilingual"`: its default English preprocessing strips
Estonian accented letters before vectorization. This was the source of many malformed
keywords in the original results, rather than necessarily a transcription error.

In [ ]:
import json
from dataclasses import asdict
from importlib.metadata import version

import numpy as np
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, silhouette_score
from umap import UMAP

ET_STOPWORDS = {
    "aga", "ei", "ega", "et", "eks", "ja", "jah", "ju", "juba",
    "ka", "kas", "kes", "kui", "kuidas", "kõik", "ma", "me", "meie",
    "meil", "mida", "midagi", "mingi", "mis", "miks", "mina", "minu",
    "mul", "mulle", "nad", "nagu", "neid", "nende", "nii", "niimoodi",
    "no", "noh", "nüüd", "on", "oli", "ole", "olen", "oleks", "oma",
    "peab", "pole", "sa", "saan", "saab", "see", "seda", "selle",
    "siin", "siis", "ta", "te", "tegelikult", "vi", "või", "veel",
    "väga", "ära", "üks", "onju",
}
ET_STOPWORDS.update({
    # ASR-i vigased ja üldised eestikeelsed vormid
    "kik", "palju", "seal", "nd", "vga", "teha", "niteks",
    "need", "vibolla", "ra", "lihtsalt", "kohta", "sellest",
    "kuidagi", "keegi", "inimesed", "niielda", "tna",
    "oleme", "hetkel", "praegu", "okei", "umbes", "sinna",
    "ikkagi",

    # Ingliskeelsed stoppsõnad
    "the", "and", "you", "that", "to", "of", "is", "like",
    "we", "in", "it", "for", "on", "with", "as", "are",
    "be", "this", "was", "have", "they", "not", "can",
    "from", "at", "or",
})

ALL_STOPWORDS = sorted(
    ET_STOPWORDS
    | set(ENGLISH_STOP_WORDS)
    | {
        "selles", "kus", "mitte", "ks", "hsti", "arvan",
        "selline", "olla", "sest", "asi", "tea", "aru",
        "vlja", "mttes", "ehk", "tlekski",
    }
)

# Keep vocabulary filtering identical across candidates.
# Preserve potentially meaningful terms (people, time, quantities) removed previously.
ALL_STOPWORDS = sorted(set(ALL_STOPWORDS) - {"inimesed", "praegu", "hetkel", "palju"})

FINE_CANDIDATES = {
    "eom-10-3": dict(n_neighbors=15, min_cluster_size=10, min_samples=3, method="eom"),
    "leaf-10-3": dict(n_neighbors=15, min_cluster_size=10, min_samples=3, method="leaf"),
    "leaf-6-2": dict(n_neighbors=15, min_cluster_size=6, min_samples=2, method="leaf"),
    "leaf-local-6-2": dict(n_neighbors=10, min_cluster_size=6, min_samples=2, method="leaf"),
}
FINE_SEEDS = (42, 7)
SELECTED_CANDIDATE = "leaf-6-2"  # Starting point; review the evidence below before choosing.
FINE_PAIRS = tuple((mode, key) for mode in SEGMENTATION_MODES for key in MODEL_KEYS)


def build_fine_model(options, seed, count):
    # Tiny offline fixtures need smaller settings, not production hyperparameters.
    small = DRY_RUN_WITH_FAKE_EMBEDDINGS
    return BERTopic(
        language="multilingual",
        embedding_model=None,
        umap_model=UMAP(
            n_neighbors=min(options["n_neighbors"], max(2, count - 1)),
            n_components=2 if small else 5,
            min_dist=0.0, metric="cosine", random_state=seed,
            init="random",
        ),
        hdbscan_model=HDBSCAN(
            min_cluster_size=2 if small else options["min_cluster_size"],
            min_samples=1 if small else options["min_samples"],
            metric="euclidean", cluster_selection_method=options["method"],
            prediction_data=True,
        ),
        vectorizer_model=CountVectorizer(
            stop_words=ALL_STOPWORDS, ngram_range=(1, 2), min_df=1,
            strip_accents=None,
        ),
        ctfidf_model=ClassTfidfTransformer(
            bm25_weighting=True, reduce_frequent_words=True,
        ),
        top_n_words=15,
        # 1-D assigned-cluster strength is enough; avoid a large soft-membership matrix.
        calculate_probabilities=False,
        nr_topics=None,
    )


## Compare granularity, coverage and seed sensitivity

The original silhouette is measured in the UMAP space used to form the clusters and can look
optimistic. Also measure cosine silhouette and within-topic similarity in the original embeddings.
Treat all scores as diagnostics: maximizing topic count or silhouette does not establish meaning.
Review representative passages, weak members, outliers and same-talk fragments.
Seed agreement excludes noise in both runs, with the retained fraction reported separately;
a low retained fraction can hide unstable assignment of difficult passages.

In [ ]:
fine_models = {}
fine_metrics = []
for mode, model_key in FINE_PAIRS:
    pair_passages = passage_sets[mode]
    documents = [p.text for p in pair_passages]
    rows = np.asarray(embedding_runs[(mode, model_key)].embeddings)
    normalized = rows / np.maximum(np.linalg.norm(rows, axis=1, keepdims=True), 1e-12)
    for candidate, options in FINE_CANDIDATES.items():
        for seed in FINE_SEEDS:
            model = build_fine_model(options, seed, len(documents))
            labels, strength = model.fit_transform(documents, embeddings=rows)
            labels = np.asarray(labels)
            fine_models[(mode, model_key, candidate, seed)] = model
            sizes = pd.Series(labels[labels != -1]).value_counts()
            valid = labels != -1
            silhouette = None
            if 1 < len(sizes) < int(valid.sum()):
                try:
                    silhouette = float(silhouette_score(
                        normalized[valid], labels[valid], metric="cosine",
                        sample_size=min(2000, int(valid.sum())), random_state=42,
                    ))
                except ValueError:
                    pass  # A sampled subset can contain only one cluster.
            similarities = []
            for topic in sizes.index:
                members = normalized[labels == topic]
                center = members.mean(axis=0)
                center /= max(np.linalg.norm(center), 1e-12)
                similarities.extend((members @ center).tolist())
            fine_metrics.append({
                "segmentation": mode, "embedding_model": model_key,
                "candidate": candidate, "seed": seed, **options,
                "topic_count": len(sizes), "outlier_fraction": float((~valid).mean()),
                "median_topic_size": float(sizes.median()) if len(sizes) else None,
                "largest_topic_fraction": float(sizes.max() / len(labels)) if len(sizes) else None,
                "original_cosine_silhouette": silhouette,
                "mean_member_centroid_cosine": float(np.mean(similarities)) if similarities else None,
            })
            print(mode, model_key, candidate, seed, "topics:", len(sizes), flush=True)

fine_metrics_frame = pd.DataFrame(fine_metrics)
fine_metrics_frame.to_csv(RESULT_ROOT / "fine-metrics.csv", index=False)
display(fine_metrics_frame)

stability_rows = []
for mode, key in FINE_PAIRS:
    for candidate in FINE_CANDIDATES:
        a = np.asarray(fine_models[(mode, key, candidate, FINE_SEEDS[0])].topics_)
        b = np.asarray(fine_models[(mode, key, candidate, FINE_SEEDS[1])].topics_)
        shared = (a != -1) & (b != -1)
        stability_rows.append({
            "segmentation": mode, "embedding_model": key, "candidate": candidate,
            "shared_inlier_fraction": float(shared.mean()),
            "outlier_status_agreement": float(np.mean((a == -1) == (b == -1))),
            "inlier_ARI": float(adjusted_rand_score(a[shared], b[shared]))
                if shared.sum() > 1 else None,
        })
stability_frame = pd.DataFrame(stability_rows)
stability_frame.to_csv(RESULT_ROOT / "seed-stability.csv", index=False)
display(stability_frame)


## Export detailed talk inventories and manual review

Each chosen model keeps its native topic IDs. Exports include all passage text and audio links,
cluster membership strength (not calibrated semantic confidence), representative examples,
weak members and outliers. Talk profiles use the union of time intervals to avoid double-counting
fixed-window overlap. Coverage of different topics may overlap in fixed mode.

Review `topic-review.csv` before treating labels as a final taxonomy. The hierarchy is a merge
proposal only; it never changes the fine model. Keep an explicit fine-to-merged mapping later.
Every candidate and seed is exported so selection can be revised without rerunning clustering.

In [ ]:
def interval_union_seconds(intervals):
    total, end = 0.0, None
    for start, stop in sorted(intervals):
        total += max(0.0, stop - max(start, end if end is not None else start))
        end = stop if end is None else max(end, stop)
    return total


for (mode, key, candidate, seed), model in fine_models.items():
    destination = RESULT_ROOT / f"{mode}-{key}" / candidate / f"seed-{seed}"
    destination.mkdir(parents=True, exist_ok=True)
    passages = passage_sets[mode]
    labels = np.asarray(model.topics_)
    strengths = np.asarray(model.probabilities_) if model.probabilities_ is not None else None
    assignments = pd.DataFrame([
        {
            "passage_id": p.passage_id, "episode_id": p.episode_id, "title": p.title,
            "duplicate_episode_ids": json.dumps(p.duplicate_episode_ids),
            "audio_sha256": p.audio_sha256,
            "start_seconds": p.start_seconds, "end_seconds": p.end_seconds,
            "topic_id": int(labels[i]),
            "membership_strength": float(strengths[i])
                if strengths is not None and labels[i] != -1 else None,
            "text": p.text, "audio_link": p.timestamped_audio_url,
        }
        for i, p in enumerate(passages)
    ])
    assignments.to_csv(destination / "passage-assignments.csv", index=False)
    assignments.query("topic_id == -1").to_csv(destination / "outliers.csv", index=False)
    info = model.get_topic_info().copy()
    info.to_csv(destination / "topics.csv", index=False)
    review_rows, profiles = [], []
    for topic, group in assignments.groupby("topic_id"):
        if topic == -1:
            continue
        representatives = model.get_representative_docs(int(topic)) or []
        representative_rows = group[group.text.isin(representatives)]
        weak = group.nsmallest(3, "membership_strength")
        review_rows.append({
            "topic_id": topic, "passage_count": len(group),
            "talk_count": group.episode_id.nunique(),
            "keywords": json.dumps(model.get_topic(int(topic)), ensure_ascii=False),
            "representatives": representative_rows.to_json(orient="records", force_ascii=False),
            "weak_members": weak.to_json(orient="records", force_ascii=False),
            "verdict": "", "label": "", "merge_into": "", "notes": "",
        })
    # Include -1 explicitly so uncovered content is visible in every talk profile.
    for (episode_id, topic), group in assignments.groupby(["episode_id", "topic_id"]):
        whole = assignments[assignments.episode_id == episode_id]
        covered = interval_union_seconds(zip(group.start_seconds, group.end_seconds))
        available = interval_union_seconds(zip(whole.start_seconds, whole.end_seconds))
        profiles.append({
            "episode_id": episode_id, "title": group.title.iloc[0], "topic_id": topic,
            "passage_count": len(group), "covered_seconds": covered,
            "fraction_of_analyzed_talk": covered / available if available else None,
        })
    pd.DataFrame(review_rows).to_csv(destination / "topic-review.csv", index=False)
    pd.DataFrame(profiles).to_csv(destination / "talk-topics.csv", index=False)
    embedding = embedding_runs[(mode, key)]
    manifest = {
        "segmentation": mode, "embedding_model": key, "candidate": candidate,
        "parameters": FINE_CANDIDATES[candidate], "seed": seed,
        "embedding_model_id": embedding.model_id, "embedding_revision": embedding.model_revision,
        "embedding_dimension": embedding.dimension,
        "passage_ids": list(embedding.passage_ids),
        "segmentation_parameters": asdict(FIXED_CHUNKING if mode == "fixed" else SEMANTIC_CONFIG),
        "language": "multilingual", "stopwords": ALL_STOPWORDS, "ngram_range": [1, 2],
        "bm25_weighting": True, "reduce_frequent_words": True,
        "top_n_words": 15, "n_components": 2 if DRY_RUN_WITH_FAKE_EMBEDDINGS else 5,
        "dry_run": DRY_RUN_WITH_FAKE_EMBEDDINGS,
        "versions": {name: version(name) for name in ["bertopic", "hdbscan", "umap-learn", "numpy", "scikit-learn"]},
    }
    (destination / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    if candidate == SELECTED_CANDIDATE and seed == FINE_SEEDS[0] and not DRY_RUN_WITH_FAKE_EMBEDDINGS:
        # Explicit trusted-local snapshot retains clustering state for later controlled merges.
        model.save(str(destination / "fine-model.pickle"), serialization="pickle", save_embedding_model=False)
        if len(set(labels) - {-1}) >= 2:
            hierarchy = model.hierarchical_topics([p.text for p in passages], use_ctfidf=True)
            hierarchy.to_csv(destination / "merge-proposals.csv", index=False)
            model.visualize_hierarchy(hierarchical_topics=hierarchy).write_html(
                destination / "hierarchy.html", include_plotlyjs=True,
            )

print(f"Detailed exports: {RESULT_ROOT}")
print(
    "SEGMENTATION_NOTEBOOK_OK "
    f"modes={','.join(SEGMENTATION_MODES)} "
    f"runs={len(runs)} boundary_figures={len(boundary_figures)}"
)
